In [1]:
import os
%pwd

'/home/amit/Kidney-Disease-Classification-MLOps/research'

In [2]:
os.chdir('../')
%pwd

'/home/amit/Kidney-Disease-Classification-MLOps'

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [5]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

2026-07-27 17:45:02.632047: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-27 17:45:02.789286: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-27 17:45:02.791631: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-27 17:45:05.397764: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [7]:

import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [8]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [11]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-07-27 17:55:33,971: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-07-27 17:55:33,982: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-27 17:55:33,985: INFO: common: created directory at: artifacts]
[2026-07-27 17:55:33,987: INFO: common: created directory at: artifacts/training]


Found 256 images belonging to 2 classes.
Found 1024 images belonging to 2 classes.
Epoch 1/10


2026-07-27 17:55:35.918362: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


64/64 [==============================] - ETA: 0s - loss: 14.2013 - accuracy: 0.5107

2026-07-27 18:00:12.063012: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


64/64 [==============================] - 343s 5s/step - loss: 14.2013 - accuracy: 0.5107 - val_loss: 17.3048 - val_accuracy: 0.5000
Epoch 2/10
64/64 [==============================] - 340s 5s/step - loss: 9.6856 - accuracy: 0.6055 - val_loss: 0.7061 - val_accuracy: 0.7852
Epoch 3/10
64/64 [==============================] - 334s 5s/step - loss: 6.4135 - accuracy: 0.6748 - val_loss: 0.9201 - val_accuracy: 0.8867
Epoch 4/10
64/64 [==============================] - 275s 4s/step - loss: 2.0905 - accuracy: 0.8330 - val_loss: 0.7381 - val_accuracy: 0.8516
Epoch 5/10
64/64 [==============================] - 276s 4s/step - loss: 2.1020 - accuracy: 0.8350 - val_loss: 0.9398 - val_accuracy: 0.8516
Epoch 6/10
64/64 [==============================] - 281s 4s/step - loss: 2.9121 - accuracy: 0.8203 - val_loss: 0.8772 - val_accuracy: 0.8516
Epoch 7/10
64/64 [==============================] - 343s 5s/step - loss: 0.1537 - accuracy: 0.9639 - val_loss: 4.0577 - val_accuracy: 0.6875
Epoch 8/10
64/64 [====